> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [ ]:
# CP_20 — Round 9
# Check the Python environment and locate the project data.

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_path = Path.cwd()

# Use the selected notebook kernel; no machine-specific path is required.
expected_python = Path(sys.executable)

initial_data_path = project_path / "initial_data"
feedback_path = project_path / "week_8"

inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

completed_rounds = 8
submission_round = 9

correct_kernel = (
    Path(sys.executable).resolve() == expected_python.resolve()
)

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("Project folder:", project_path)

print("\nSelected interpreter check:", correct_kernel)
print("Project folder found:", project_path.is_dir())
print("Initial data folder found:", initial_data_path.is_dir())
print("Feedback folder found:", feedback_path.is_dir())
print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())

print("\nExpected completed rounds:", completed_rounds)
print("Round to prepare:", submission_round)

assert correct_kernel, (
    "Select the notebook kernel corresponding to: "
    f"{expected_python}"
)
assert project_path.is_dir(), (
    f"Project folder not found: {project_path}"
)
assert initial_data_path.is_dir(), (
    f"Initial data folder not found: {initial_data_path}"
)
assert feedback_path.is_dir(), (
    f"Feedback folder not found: {feedback_path}"
)
assert inputs_path.is_file(), (
    f"Inputs file not found: {inputs_path}"
)
assert outputs_path.is_file(), (
    f"Outputs file not found: {outputs_path}"
)

print("\nEnvironment and file checks passed.")
print("Ready to load and validate the eight feedback rounds.")

In [3]:
# CP_20 — Round 9
# Load and validate the initial data and eight completed feedback rounds.

import ast
import re
import numpy as np


def read_feedback_rounds(path):
    """Read cumulative feedback safely without executing file contents."""
    text = path.read_text(encoding="utf-8-sig").strip()

    # Convert NumPy-style array([...]) representations into plain lists.
    text = re.sub(r"array\((\[[^\[\]]*\])\)", r"\1", text)

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(
                f"Unexpected content in {path.name}: "
                "expected one list per feedback round."
            )

        rounds.append(ast.literal_eval(node.value))

    return rounds


# Check that the first code cell has been executed.
required_variables = [
    "initial_data_path",
    "inputs_path",
    "outputs_path",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the first code cell before this cell. "
        f"Missing variables: {', '.join(missing_variables)}"
    )


expected_dimensions = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

expected_initial_counts = {
    1: 10,
    2: 10,
    3: 15,
    4: 30,
    5: 20,
    6: 20,
    7: 30,
    8: 40,
}


# Load the cumulative feedback files.
input_rounds = read_feedback_rounds(inputs_path)
output_rounds = read_feedback_rounds(outputs_path)

assert len(input_rounds) == completed_rounds, (
    f"Expected {completed_rounds} input rounds, "
    f"but found {len(input_rounds)}."
)

assert len(output_rounds) == completed_rounds, (
    f"Expected {completed_rounds} output rounds, "
    f"but found {len(output_rounds)}."
)


# Validate every feedback input–output pair.
for round_number, (round_inputs, round_outputs) in enumerate(
    zip(input_rounds, output_rounds),
    start=1,
):
    assert len(round_inputs) == 8, (
        f"Round {round_number}: expected inputs for eight functions."
    )

    assert len(round_outputs) == 8, (
        f"Round {round_number}: expected outputs for eight functions."
    )

    for function, dimension in expected_dimensions.items():
        x = np.asarray(round_inputs[function - 1], dtype=float)
        y = np.asarray(round_outputs[function - 1], dtype=float)

        assert x.shape == (dimension,), (
            f"Round {round_number}, Function {function}: "
            f"expected {dimension} coordinates, found shape {x.shape}."
        )

        assert y.ndim == 0, (
            f"Round {round_number}, Function {function}: "
            "expected one scalar output."
        )

        assert np.all(np.isfinite(x)) and np.isfinite(y).item(), (
            f"Round {round_number}, Function {function}: "
            "non-finite values detected."
        )

        assert np.all((x >= 0) & (x < 1)), (
            f"Round {round_number}, Function {function}: "
            "feedback coordinates must satisfy 0 <= x < 1."
        )


# Rebuild the datasets from the original files on every execution.
# This prevents appending the same feedback twice when rerunning the cell.
loaded_initial_data = {}
loaded_updated_data = {}

for function, dimension in expected_dimensions.items():
    function_path = initial_data_path / f"function_{function}"

    initial_inputs_file = function_path / "initial_inputs.npy"
    initial_outputs_file = function_path / "initial_outputs.npy"

    assert initial_inputs_file.is_file(), (
        f"File not found: {initial_inputs_file}"
    )

    assert initial_outputs_file.is_file(), (
        f"File not found: {initial_outputs_file}"
    )

    X_initial = np.asarray(
        np.load(initial_inputs_file, allow_pickle=False),
        dtype=float,
    )

    y_initial = np.asarray(
        np.load(initial_outputs_file, allow_pickle=False),
        dtype=float,
    ).reshape(-1)

    expected_count = expected_initial_counts[function]

    assert X_initial.shape == (expected_count, dimension), (
        f"Function {function}: unexpected initial input shape "
        f"{X_initial.shape}."
    )

    assert y_initial.shape == (expected_count,), (
        f"Function {function}: unexpected initial output shape "
        f"{y_initial.shape}."
    )

    assert np.all(np.isfinite(X_initial)), (
        f"Function {function}: non-finite initial inputs."
    )

    assert np.all(np.isfinite(y_initial)), (
        f"Function {function}: non-finite initial outputs."
    )

    X_feedback = np.asarray(
        [values[function - 1] for values in input_rounds],
        dtype=float,
    )

    y_feedback = np.asarray(
        [values[function - 1] for values in output_rounds],
        dtype=float,
    )

    loaded_initial_data[function] = {
        "X": X_initial.copy(),
        "y": y_initial.copy(),
    }

    loaded_updated_data[function] = {
        "X": np.vstack([X_initial, X_feedback]),
        "y": np.concatenate([y_initial, y_feedback]),
    }


# Make the datasets available only after all eight pass validation.
initial_data = loaded_initial_data
updated_data = loaded_updated_data

for function, dimension in expected_dimensions.items():
    initial_count = len(initial_data[function]["y"])
    updated_count = len(updated_data[function]["y"])

    print(
        f"Function {function}: "
        f"{initial_count} → {updated_count} observations "
        f"| {dimension} variables"
    )

print(f"\nValidated feedback rounds: {len(input_rounds)}")
print(f"Round to prepare: {submission_round}")
print("All eight datasets loaded and validated successfully.")

Function 1: 10 → 18 observations | 2 variables
Function 2: 10 → 18 observations | 2 variables
Function 3: 15 → 23 observations | 3 variables
Function 4: 30 → 38 observations | 4 variables
Function 5: 20 → 28 observations | 4 variables
Function 6: 20 → 28 observations | 5 variables
Function 7: 30 → 38 observations | 6 variables
Function 8: 40 → 48 observations | 8 variables

Validated feedback rounds: 8
Round to prepare: 9
All eight datasets loaded and validated successfully.


### Data loading and validation

The initial observations were combined with eight completed feedback rounds for each function. The updated datasets contain 18, 18, 23, 38, 28, 28, 38 and 48 observations, respectively.

The checks confirmed the expected number of rounds, input dimensions, finite numerical values and permitted feedback coordinate ranges. Each dataset was rebuilt from its source files, preventing duplicate appends when rerunning the cell.

The next step evaluates whether the latest observations improved the best previously observed outputs. All eight functions are treated as maximisation problems.

In [4]:
# Review round 8 results and track improvements across all completed rounds.

import numpy as np
import pandas as pd
from IPython.display import display

latest_round = completed_rounds
feedback_outputs = np.asarray(output_rounds, dtype=float)

assert feedback_outputs.shape == (completed_rounds, 8), (
    "Expected one output per function for every completed round."
)

latest_rows = []
history_rows = []

for function in range(1, 9):
    initial_best = float(np.max(initial_data[function]["y"]))
    best_so_far = initial_best

    for round_index in range(completed_rounds):
        round_number = round_index + 1
        observed_output = float(
            feedback_outputs[round_index, function - 1]
        )

        best_before = best_so_far

        # A larger output is better, including when outputs are negative.
        improved = observed_output > best_before
        improvement = max(0.0, observed_output - best_before)
        best_so_far = max(best_before, observed_output)

        history_rows.append({
            "Round": round_number,
            "Function": function,
            "Observed output": observed_output,
            "Best before round": best_before,
            "Improved": improved,
            "Improvement": improvement,
            "Best after round": best_so_far,
        })

        if round_number == latest_round:
            latest_rows.append({
                "Function": function,
                f"Best before round {latest_round}": best_before,
                f"Round {latest_round} output": observed_output,
                f"Round {latest_round} improved": improved,
                "Improvement": improvement,
                "Current best": best_so_far,
            })

    # Confirm that the history agrees with the combined dataset.
    assert best_so_far == float(np.max(updated_data[function]["y"])), (
        f"Function {function}: inconsistent best observed output."
    )

improvement_history = pd.DataFrame(history_rows)

round_results = (
    pd.DataFrame(latest_rows)
    .set_index("Function")
)

numeric_columns = [
    f"Best before round {latest_round}",
    f"Round {latest_round} output",
    "Improvement",
    "Current best",
]

display(
    round_results.style.format({
        column: "{:.6e}" for column in numeric_columns
    })
)

print()

for round_number in range(1, completed_rounds + 1):
    improved_functions = improvement_history.loc[
        (improvement_history["Round"] == round_number)
        & improvement_history["Improved"],
        "Function",
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

,Best before round 8,Round 8 output,Round 8 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,4.639479e-156,False,0.000000e+00,7.710875e-16
2,6.582395e-01,5.476083e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-1.263641e-01,False,0.000000e+00,-3.096391e-02
4,4.098217e-01,6.152729e-01,True,2.054512e-01,6.152729e-01
5,8.343789e+03,8.510289e+03,True,1.665003e+02,8.510289e+03
6,-2.424310e-01,-4.641445e-01,False,0.000000e+00,-2.424310e-01
7,2.171300e+00,2.374721e+00,True,2.034209e-01,2.374721e+00
8,9.922955e+00,9.915943e+00,False,0.000000e+00,9.922955e+00



Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3
Functions improved in round 7: [5] | Count: 1
Functions improved in round 8: [4, 5, 7] | Count: 3


### Round 8 — Observed improvements

Round 8 improved the best previously observed outputs for Functions 4, 5 and 7. The remaining five functions did not exceed their existing best results.

The number of functions improving in each round was 5, 2, 2, 2, 4, 3, 1 and 3, respectively. Progress has therefore varied across rounds and functions. Function 5 improved in three consecutive rounds, while Functions 4 and 7 improved again after rounds without a new best result.

These counts describe how broadly improvements occurred, but do not capture their magnitude or establish that one strategy is consistently better. Outputs also have different scales, so raw improvements should not be added across functions.

Observations that did not improve the current best still provide evidence about the response surface. Before selecting round 9 queries, the next step is to compare the predictions recorded before round 8 with the actual outputs, paying attention to prediction errors relative to the estimated uncertainty.

In [5]:
# Compare the predictions recorded before round 8 with its observed outputs.
# Historical predictions are copied from the displayed CP_19 results.
# They are rounded values, so the calculated errors are approximate.
# No model is refitted in this cell.

import numpy as np
import pandas as pd
from IPython.display import display

review_round = 8

assert completed_rounds >= review_round, (
    "Round 8 feedback is required for this comparison."
)

# Query, GP predicted mean, GP predicted standard deviation,
# and auxiliary prediction where available.
round_8_predictions = {
    1: {
        "query": "0.208681-0.000255",
        "mean": -1.899534e-04,
        "std": 8.939013e-04,
        "auxiliary": np.nan,
    },
    2: {
        "query": "0.698207-0.000240",
        "mean": 6.023600e-01,
        "std": 7.089054e-02,
        "auxiliary": np.nan,
    },
    3: {
        "query": "0.991779-0.022828-0.094305",
        "mean": -5.384088e-02,
        "std": 5.364524e-02,
        "auxiliary": np.nan,
    },
    4: {
        "query": "0.424455-0.424454-0.381378-0.414254",
        "mean": 2.913625e-01,
        "std": 1.338038e-01,
        "auxiliary": np.nan,
    },
    5: {
        "query": "0.997017-0.998912-0.999252-0.996899",
        "mean": 7.760426e+03,
        "std": 3.830841e+02,
        "auxiliary": 7.871547e+03,
    },
    6: {
        "query": "0.424979-0.403855-0.530055-0.676776-0.000314",
        "mean": -3.707889e-01,
        "std": 1.302793e-01,
        "auxiliary": np.nan,
    },
    7: {
        "query": (
            "0.019096-0.226554-0.575958-"
            "0.153734-0.332502-0.733585"
        ),
        "mean": 2.086414e+00,
        "std": 1.189489e-01,
        "auxiliary": 8.946228e-01,
    },
    8: {
        "query": (
            "0.149147-0.311840-0.192931-0.145640-"
            "0.987373-0.581956-0.118912-0.792680"
        ),
        "mean": 9.914491e+00,
        "std": 4.599624e-02,
        "auxiliary": np.nan,
    },
}

prediction_review_rows = []

for function in range(1, 9):
    prediction = round_8_predictions[function]

    recorded_query = np.asarray(
        [float(value) for value in prediction["query"].split("-")],
        dtype=float,
    )

    evaluated_query = np.asarray(
        input_rounds[review_round - 1][function - 1],
        dtype=float,
    )

    assert recorded_query.shape == evaluated_query.shape, (
        f"Function {function}: query dimensions do not match."
    )

    assert np.allclose(
        recorded_query,
        evaluated_query,
        rtol=0,
        atol=1e-9,
    ), (
        f"Function {function}: the historical prediction does not "
        "correspond to the input evaluated in round 8."
    )

    observed = float(
        output_rounds[review_round - 1][function - 1]
    )
    mean = prediction["mean"]
    std = prediction["std"]
    auxiliary = prediction["auxiliary"]

    assert np.isfinite(mean) and np.isfinite(std) and std > 0, (
        f"Function {function}: invalid historical prediction."
    )

    residual = observed - mean
    absolute_error = abs(residual)

    auxiliary_error = (
        abs(observed - auxiliary)
        if np.isfinite(auxiliary)
        else np.nan
    )

    prediction_review_rows.append({
        "Function": function,
        "Predicted mean": mean,
        "Predicted std": std,
        "Observed output": observed,
        "Observed minus predicted": residual,
        "Absolute error / predicted std": absolute_error / std,
        "GP absolute error": absolute_error,
        "Auxiliary prediction": auxiliary,
        "Auxiliary absolute error": auxiliary_error,
    })

round_8_prediction_review = (
    pd.DataFrame(prediction_review_rows)
    .set_index("Function")
)

formatting = {
    column: "{:.6e}"
    for column in round_8_prediction_review.columns
}
formatting["Absolute error / predicted std"] = "{:.2f}"

display(
    round_8_prediction_review.style.format(
        formatting,
        na_rep="—",
    )
)

print("All eight historical queries match the round 8 feedback inputs.")
print("Errors use rounded historical predictions from CP_19.")
print(
    "Error / predicted std is a diagnostic for these individual "
    "observations, not a complete uncertainty calibration assessment."
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-1.899534e-04,8.939013e-04,4.639479e-156,1.899534e-04,0.21,1.899534e-04,—,—
2,6.023600e-01,7.089054e-02,5.476083e-01,-5.475168e-02,0.77,5.475168e-02,—,—
3,-5.384088e-02,5.364524e-02,-1.263641e-01,-7.252317e-02,1.35,7.252317e-02,—,—
4,2.913625e-01,1.338038e-01,6.152729e-01,3.239104e-01,2.42,3.239104e-01,—,—
5,7.760426e+03,3.830841e+02,8.510289e+03,7.498634e+02,1.96,7.498634e+02,7.871547e+03,6.387424e+02
6,-3.707889e-01,1.302793e-01,-4.641445e-01,-9.335562e-02,0.72,9.335562e-02,—,—
7,2.086414e+00,1.189489e-01,2.374721e+00,2.883074e-01,2.42,2.883074e-01,8.946228e-01,1.480099e+00
8,9.914491e+00,4.599624e-02,9.915943e+00,1.451825e-03,0.03,1.451825e-03,—,—


All eight historical queries match the round 8 feedback inputs.
Errors use rounded historical predictions from CP_19.
Error / predicted std is a diagnostic for these individual observations, not a complete uncertainty calibration assessment.


### Round 8 — Prediction errors and uncertainty

This comparison uses predictions recorded before round 8 and the corresponding observed outputs. All eight query coordinates match the feedback inputs. Historical predictions were copied from the displayed CP_19 table, so the calculated errors are approximate.

Functions 4 and 7 exceeded their GP predictions by approximately 2.42 predictive standard deviations. Both observations established new best results. These discrepancies suggest that uncertainty may have been underestimated at these particular locations, but one observation per function cannot establish overall calibration.

Function 5 also produced a new best result, reaching approximately 8,510.29. The GP underestimated this output by 749.86, equivalent to 1.96 predictive standard deviations. The auxiliary neural network was closer, with an absolute error of 638.74. This supports retaining it as a comparison model, although this single result does not demonstrate consistent superiority.

For Function 7, the GP was substantially more accurate than the auxiliary linear regression: their absolute errors were approximately 0.29 and 1.48, respectively. The disagreement highlights the limitations of using a simple linear model to assess this promising region.

Function 8 had the smallest error relative to predicted uncertainty, at approximately 0.03 standard deviations. However, its observed output did not improve the existing best. Accurate prediction and successful optimisation are related but distinct objectives.

Functions 1, 2 and 6 had errors below one predictive standard deviation, while Function 3 had an error of approximately 1.35. For Function 1, the small ratio does not establish that the model can identify better regions: the observed output remains extremely close to zero.

These results support updating the models with all eight new observations and continuing to compare predictions, uncertainty and actual improvements. Predictive uncertainty should inform query selection without being treated as a guarantee.

In [6]:
# CP_20 — Round 9
# Refit the Gaussian Processes using all observations through round 8.
# Retain the kernel configurations used in CP_19.

import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

assert "updated_data" in globals(), (
    "Run the data-loading cell before fitting the models."
)

assert completed_rounds == 8 and submission_round == 9, (
    "This cell prepares round 9 using eight completed feedback rounds."
)

# Configurations retained from CP_19.
gp_configurations = {
    1: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    2: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    3: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    4: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    5: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    6: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    7: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    8: {
        "name": "Matern_2.5_wider",
        "nu": 2.5,
        "upper_bound": 10.0,
    },
}

fitted_models = {}
fit_rows = []
warning_rows = []

for function in range(1, 9):
    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)

    configuration = gp_configurations[function]

    assert X.ndim == 2 and y.shape == (X.shape[0],), (
        f"Function {function}: incompatible input and output shapes."
    )
    assert np.all(np.isfinite(X)) and np.all(np.isfinite(y)), (
        f"Function {function}: non-finite training data."
    )

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.full(X.shape[1], 0.2),
            length_scale_bounds=(
                0.02,
                configuration["upper_bound"],
            ),
            nu=configuration["nu"],
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=42,
    )

    start_time = time.perf_counter()

    # Preserve fitting warnings for inspection below.
    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = time.perf_counter() - start_time

    fitted_models[function] = model

    fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "GP configuration": configuration["name"],
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": (
            model.log_marginal_likelihood_value_
        ),
        "GP fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

# Make all eight fitted models available to subsequent cells.
gp_models = fitted_models

gp_fit_summary = (
    pd.DataFrame(fit_rows)
    .set_index("Function")
)

gp_fit_warnings = pd.DataFrame(
    warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    gp_fit_summary.style.format({
        "Log-marginal likelihood": "{:.6f}",
        "GP fit seconds": "{:.3f}",
    })
)

print(
    "\nAll eight Gaussian Processes were fitted "
    "using observations through round 8."
)
print(
    "Kernel configurations were retained from CP_19; "
    "continuous kernel parameters were re-estimated."
)

if gp_fit_warnings.empty:
    print("\nNo fitting warnings were recorded.")
else:
    print(f"\nRecorded fitting warnings: {len(gp_fit_warnings)}")

    with pd.option_context("display.max_colwidth", None):
        display(gp_fit_warnings)

,Observations,GP configuration,Fitted kernel,Log-marginal likelihood,GP fit seconds
Function,,,,,
1,18,Matern_1.5_original,"1.06**2 * Matern(length_scale=[0.02, 2], nu=1.5) + WhiteKernel(noise_level=1e-10)",-24.264326,0.265
2,18,Matern_1.5_wider,"1.03**2 * Matern(length_scale=[0.0985, 10], nu=1.5) + WhiteKernel(noise_level=0.0559)",-15.885610,0.103
3,23,Matern_1.5_original,"1.25**2 * Matern(length_scale=[0.53, 2, 0.166], nu=1.5) + WhiteKernel(noise_level=0.00274)",-27.542041,0.156
4,38,Matern_1.5_wider,"4.35**2 * Matern(length_scale=[3.54, 3.29, 3.38, 3.42], nu=1.5) + WhiteKernel(noise_level=8.02e-05)",0.495097,0.220
5,28,Matern_1.5_original,"1.2**2 * Matern(length_scale=[1.06, 1.84, 0.63, 2], nu=1.5) + WhiteKernel(noise_level=0.000103)",-10.186971,0.145
6,28,Matern_2.5_original,"1.71**2 * Matern(length_scale=[0.993, 1.36, 1.42, 1.26, 2], nu=2.5) + WhiteKernel(noise_level=0.0367)",-22.454309,0.187
7,38,Matern_2.5_original,"0.681**2 * Matern(length_scale=[0.903, 2, 2, 0.177, 0.153, 0.168], nu=2.5) + WhiteKernel(noise_level=8.84e-10)",-30.975313,0.193
8,48,Matern_2.5_wider,"2.97**2 * Matern(length_scale=[3.09, 4.69, 2.48, 5.28, 10, 4.97, 3.19, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",12.514389,0.418



All eight Gaussian Processes were fitted using observations through round 8.
Kernel configurations were retained from CP_19; continuous kernel parameters were re-estimated.

Recorded fitting warnings: 12


,Function,Category,Message
0,1,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
1,1,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
2,1,ConvergenceWarning,The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
3,2,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
4,3,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
5,5,ConvergenceWarning,The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
6,6,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
7,7,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
8,7,ConvergenceWarning,The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
9,8,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.


### Round 9 — Gaussian Process fitting and diagnostics

All eight Gaussian Processes were successfully fitted using the initial observations and feedback from eight completed rounds. The kernel configurations from CP_19 were retained, while the length scales, signal amplitude and noise parameters were re-estimated. This step updates the existing models; it does not repeat cross-validation or establish that these configurations remain optimal.

Fitting took approximately 0.10–0.42 seconds per function, making the current approach computationally manageable for datasets containing 18–48 observations.

### Interpreting the fitting warnings

The 12 recorded warnings concern parameters reaching their permitted bounds. They did not prevent fitting, but indicate where the chosen limits may constrain the models.

- **Function 1:** The first length scale reached the lower bound of 0.02, while the second reached the upper bound of 2.0. The fitted model therefore represents much faster variation along the first coordinate. Given the limited signal in the observed outputs, this should not be interpreted as a reliable description of the true function.
- **Functions 2 and 3:** The second length scale reached its upper bound, suggesting relatively slow predicted variation along that coordinate under the current model.
- **Functions 5 and 6:** The fourth and fifth length scales, respectively, reached the upper bound of 2.0.
- **Function 7:** The second and third length scales reached 2.0, while the fourth, fifth and sixth were much shorter. This suggests greater model sensitivity to changes in the latter coordinates.
- **Function 8:** The fifth and eighth length scales reached 10.0. Their relatively weak influence in this fitted model does not establish that these variables are irrelevant.
- **Functions 1 and 8:** The noise parameter reached its lower bound. This indicates that the fitted models favour very little independent noise, but does not prove that the black-box evaluations are noise-free.

Function 4 produced no fitting warnings. Its four length scales were similar, ranging from approximately 3.29 to 3.54. However, the absence of warnings does not guarantee accurate predictions or well-calibrated uncertainty.

### Implications for query selection

The fitted parameters remain modelling assumptions supported by limited observations. In particular, the previous prediction errors for Functions 4 and 7 show why uncertainty estimates should be interpreted cautiously.

The log-marginal likelihood values describe fit under each model's assumptions. They should not be used to rank different functions or treated as evidence of improved predictive accuracy when comparing rounds with different datasets.

These models will provide predicted means and standard deviations for round 9 candidate selection. Auxiliary predictions for Functions 5 and 7 will provide an additional comparison. Boundary warnings will remain documented, and any changes to parameter bounds should be assessed through validation or sensitivity analysis rather than made solely to remove warnings.

In [7]:
# CP_20 — Round 9
# Refit the auxiliary models using observations through round 8.

import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

assert "updated_data" in globals(), (
    "Run the data-loading cell before fitting the auxiliary models."
)

# Retain the auxiliary model settings used in the previous round.
auxiliary_estimators = {
    5: (
        "NN (8,)",
        MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42,
        ),
    ),
    7: (
        "Linear regression",
        LinearRegression(),
    ),
}

fitted_auxiliary_models = {}
auxiliary_fit_rows = []
auxiliary_warning_rows = []

for function, (model_name, estimator) in auxiliary_estimators.items():
    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)

    assert X.ndim == 2 and y.shape == (X.shape[0],), (
        f"Function {function}: incompatible input and output shapes."
    )
    assert np.all(np.isfinite(X)) and np.all(np.isfinite(y)), (
        f"Function {function}: non-finite training data."
    )

    # Standardise inputs and targets during fitting.
    # Predictions are automatically returned in original output units.
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("input_scaler", StandardScaler()),
            ("model", estimator),
        ]),
        transformer=StandardScaler(),
    )

    start_time = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = time.perf_counter() - start_time

    fitted_auxiliary_models[function] = model

    auxiliary_fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Auxiliary model": model_name,
        "Fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        auxiliary_warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

auxiliary_models = fitted_auxiliary_models

auxiliary_fit_summary = (
    pd.DataFrame(auxiliary_fit_rows)
    .set_index("Function")
)

auxiliary_fit_warnings = pd.DataFrame(
    auxiliary_warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    auxiliary_fit_summary.style.format({
        "Fit seconds": "{:.3f}",
    })
)

print(
    "\nAuxiliary models fitted for Functions 5 and 7 "
    "using observations through round 8."
)
print("Predictions will be returned in the original output units.")

if auxiliary_fit_warnings.empty:
    print("No auxiliary fitting warnings were recorded.")
else:
    print(
        f"\nRecorded auxiliary fitting warnings: "
        f"{len(auxiliary_fit_warnings)}"
    )
    with pd.option_context("display.max_colwidth", None):
        display(auxiliary_fit_warnings)
        

,Observations,Auxiliary model,Fit seconds
Function,,,
5,28,"NN (8,)",0.062
7,38,Linear regression,0.011



Auxiliary models fitted for Functions 5 and 7 using observations through round 8.
Predictions will be returned in the original output units.
No auxiliary fitting warnings were recorded.


### Round 9 — Auxiliary models

The auxiliary neural network for Function 5 and linear regression for Function 7 were refitted using 28 and 38 observations, respectively. Their configurations were retained from the previous round.

Inputs and targets were standardised during fitting. Predictions are automatically transformed back into the original output units, allowing direct comparison with GP predictions and observed results.

For Function 5, the neural network provides a flexible non-linear comparison. Its round 8 absolute prediction error was approximately 638.74, compared with 749.86 for the GP. This single observation supports further comparison but does not establish that the neural network is consistently more accurate.

For Function 7, linear regression provides a simpler reference model. Its round 8 absolute error was approximately 1.48, compared with 0.29 for the GP. Its predictions will therefore be interpreted cautiously, particularly near promising regions where a linear relationship may be inadequate.

This cell performs fitting rather than a new validation comparison. Fit times and any recorded warnings describe computational behaviour, not predictive quality.

The next step compares global and local candidate points. GP predictions and uncertainty will determine UCB scores; auxiliary predictions will be reported separately to highlight agreement or disagreement between models.

In [8]:
# CP_20 — Round 9
# Compare global and local candidates using GP-UCB.

import time

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.spatial import cKDTree

assert "gp_models" in globals(), "Run the GP fitting cell first."
assert "auxiliary_models" in globals(), (
    "Run the auxiliary model fitting cell first."
)

ucb_weight = 2.0
global_candidate_count = 100_000
local_candidate_count = 10_000
local_radii = [0.10, 0.03]
prediction_batch_size = 5_000

lower_bound = 0.0
upper_bound = 0.999999
duplicate_tolerance = 1e-6

round_9_candidate_proposals = {}
candidate_rows = []
search_time_rows = []


def predict_in_batches(model, candidates, batch_size):
    """Calculate GP predictions without processing all candidates at once."""
    means = []
    standard_deviations = []

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]
        mean, std = model.predict(batch, return_std=True)
        means.append(mean)
        standard_deviations.append(std)

    return np.concatenate(means), np.concatenate(standard_deviations)


for function in range(1, 9):
    start_time = time.perf_counter()

    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)
    model = gp_models[function]

    dimension = X.shape[1]
    best_index = int(np.argmax(y))
    best_input = X[best_index]
    current_best = float(y[best_index])

    rng = np.random.default_rng(42 + function)
    observation_tree = cKDTree(X)

    candidate_sets = {
        "Global": rng.uniform(
            lower_bound,
            upper_bound,
            size=(global_candidate_count, dimension),
        )
    }

    # Local regions are boxes centred on the best observed input.
    # Bounds are restricted before sampling, avoiding clipping duplicates.
    for radius in local_radii:
        local_lower = np.maximum(lower_bound, best_input - radius)
        local_upper = np.minimum(upper_bound, best_input + radius)

        candidate_sets[f"Local {radius:.2f}"] = rng.uniform(
            local_lower,
            local_upper,
            size=(local_candidate_count, dimension),
        )

    round_9_candidate_proposals[function] = {}

    for search_name, candidates in candidate_sets.items():
        # Evaluate the same six-decimal coordinates used for submission.
        candidates = np.unique(
            np.round(candidates, 6),
            axis=0,
        )

        nearest_distances, _ = observation_tree.query(candidates, k=1)

        keep = nearest_distances > duplicate_tolerance
        candidates = candidates[keep]
        nearest_distances = nearest_distances[keep]

        assert len(candidates) > 0, (
            f"Function {function}, {search_name}: no valid candidates."
        )
        assert np.all(
            (candidates >= lower_bound) & (candidates <= upper_bound)
        ), "Candidate coordinates are outside the permitted bounds."

        means, standard_deviations = predict_in_batches(
            model,
            candidates,
            prediction_batch_size,
        )

        assert np.all(np.isfinite(means)), "Non-finite GP means."
        assert np.all(np.isfinite(standard_deviations)), (
            "Non-finite GP standard deviations."
        )

        ucb_scores = means + ucb_weight * standard_deviations
        selected_index = int(np.argmax(ucb_scores))
        query = candidates[selected_index].copy()

        auxiliary_prediction = np.nan
        if function in auxiliary_models:
            auxiliary_prediction = float(
                auxiliary_models[function].predict(
                    query.reshape(1, -1)
                )[0]
            )

        proposal = {
            "query": query,
            "mean": float(means[selected_index]),
            "std": float(standard_deviations[selected_index]),
            "ucb": float(ucb_scores[selected_index]),
            "nearest_distance": float(
                nearest_distances[selected_index]
            ),
            "auxiliary_prediction": auxiliary_prediction,
        }

        round_9_candidate_proposals[function][search_name] = proposal

        candidate_rows.append({
            "Function": function,
            "Search": search_name,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": current_best,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction,
        })

    search_time_rows.append({
        "Function": function,
        "Candidate search seconds": time.perf_counter() - start_time,
    })


round_9_candidates = pd.DataFrame(candidate_rows)
candidate_search_times = (
    pd.DataFrame(search_time_rows)
    .set_index("Function")
)

display(
    round_9_candidates.style.format(
        {
            "Current best": "{:.6e}",
            "GP mean": "{:.6e}",
            "GP std": "{:.6e}",
            "UCB": "{:.6e}",
            "Nearest observation distance": "{:.6f}",
            "Auxiliary prediction": "{:.6e}",
        },
        na_rep="—",
    )
)

display(
    candidate_search_times.style.format({
        "Candidate search seconds": "{:.3f}",
    })
)

print("\nGenerated three proposals per function: 24 proposals in total.")
print("Each proposal maximises UCB within its sampled candidate set.")
print("Final round 9 queries have not yet been selected.")

,Function,Search,Query,Current best,GP mean,GP std,UCB,Nearest observation distance,Auxiliary prediction
0,1,Global,0.257861-0.414992,7.710875e-16,-1.777581e-04,8.715443e-04,1.565331e-03,0.142548,—
1,1,Local 0.10,0.763810-0.828314,7.710875e-16,-1.172491e-04,8.283223e-04,1.539395e-03,0.086804,—
2,1,Local 0.03,0.761019-0.761965,7.710875e-16,-1.164723e-04,8.256463e-04,1.534820e-03,0.041698,—
3,2,Global,0.693314-0.998760,6.582395e-01,5.778382e-01,7.550253e-02,7.288432e-01,0.059537,—
4,2,Local 0.10,0.705628-0.001431,6.582395e-01,5.908901e-01,6.853092e-02,7.279520e-01,0.005857,—
5,2,Local 0.03,0.706744-0.000108,6.582395e-01,5.904171e-01,6.877413e-02,7.279654e-01,0.007216,—
6,3,Global,0.607473-0.016509-0.462216,-3.096391e-02,-3.625537e-02,5.902179e-02,8.178821e-02,0.383088,—
7,3,Local 0.10,0.901276-0.869534-0.481066,-3.096391e-02,-2.997549e-02,2.583190e-02,2.168832e-02,0.107771,—
8,3,Local 0.03,0.971468-0.934854-0.475738,-3.096391e-02,-3.036965e-02,1.800706e-02,5.644466e-03,0.050509,—
9,4,Global,0.492582-0.419899-0.416838-0.400862,6.152729e-01,-5.476357e-02,3.343608e-01,6.139580e-01,0.078095,—


,Candidate search seconds
Function,
1,0.474
2,0.343
3,0.376
4,0.482
5,0.469
6,0.478
7,0.565
8,0.716



Generated three proposals per function: 24 proposals in total.
Each proposal maximises UCB within its sampled candidate set.
Final round 9 queries have not yet been selected.


### Round 9 — Global and local candidate search

For each function, the search compares 100,000 uniformly sampled global candidates with 10,000 candidates from each of two local regions around the best observed input.

The local regions use coordinate-wise radii of 0.10 and 0.03, restricted to the permitted input domain. These radii define sampling boxes, not Euclidean distance limits.

Coordinates are rounded to six decimal places before prediction. Duplicate candidates within each set are removed, and points within a Euclidean distance of 1e-6 from an existing observation are excluded.

Each candidate receives a UCB score equal to the GP predicted mean plus twice its predictive standard deviation. The highest-scoring candidate from each search region is retained, producing three proposals per function. These are maxima over sampled candidates, not guaranteed maxima over the continuous domain.

Global sampling allows exploration across the domain, while local sampling provides denser coverage around the current best observation. A globally generated candidate may still lie near known observations; its search label alone does not determine how exploratory it is.

Auxiliary predictions for Functions 5 and 7 are reported for comparison and do not affect the UCB calculation. The nearest-observation distance provides additional context about each proposal, although raw distances are not directly comparable across different dimensions.

The next step compares the three proposals for each function and selects one query for round 9. Predicted means and UCB scores are model estimates, not observed outcomes or guarantees of improvement.

### Round 9 — Interpreting the candidate comparison

The highest-UCB proposals come from the global search for Functions 1, 2, 3 and 7, the local 0.03 search for Function 4, and the local 0.10 search for Functions 5, 6 and 8.

For Functions 1 and 3, the selected global candidates have lower predicted means than the local alternatives, but greater predictive uncertainty raises their UCB scores. These choices therefore place more emphasis on exploration.

For Function 2, the global proposal has only a slightly higher UCB than the local alternatives. Its second coordinate is near the upper boundary, whereas the local proposals are near the lower boundary. The small score difference should not be interpreted as strong evidence that the global proposal is superior.

Function 4 favours refinement near the best observed input. The local 0.03 candidate combines the highest predicted mean among the three proposals with sufficient uncertainty to obtain the highest UCB.

For Function 5, the local 0.10 proposal has a lower predicted mean than the local 0.03 proposal, but its higher uncertainty produces a higher UCB. The auxiliary neural network predicts a lower output than the GP, highlighting model disagreement.

For Function 6, the local 0.10 and local 0.03 UCB scores are very close. The local 0.10 score is the larger, less negative value, so it is selected under the maximisation rule.

For Function 7, the global candidate narrowly exceeds the local alternatives in UCB. The linear model predicts a substantially lower output than the GP. This disagreement remains relevant, although the linear model was less accurate at the round 8 query.

Function 8 favours the local 0.10 proposal, which has the highest predicted mean and UCB among the three candidates.

Only Function 8's selected candidate has a GP predicted mean above the current best observed output. The remaining selections rely on uncertainty to provide potential for improvement. None of these predictions guarantees a new best result.

In [9]:
# CP_20 — Round 9
# Select the highest-UCB proposal for each function.
# Validate and display the final queries for manual portal submission.

import re

import numpy as np
import pandas as pd
from IPython.display import display

assert "round_9_candidate_proposals" in globals(), (
    "Run the candidate-search cell before selecting final queries."
)
assert submission_round == 9, "This cell prepares round 9 queries."

selected_models = {}
selected_queries = {}
round_9_selected_proposals = {}
selection_rows = []

for function in range(1, 9):
    proposals = round_9_candidate_proposals[function]

    assert len(proposals) == 3, (
        f"Function {function}: expected three candidate proposals."
    )
    assert all(
        np.isfinite(proposal["ucb"])
        for proposal in proposals.values()
    ), f"Function {function}: non-finite UCB score."

    selected_search = max(
        proposals,
        key=lambda name: proposals[name]["ucb"],
    )

    proposal = proposals[selected_search]
    query = np.asarray(proposal["query"], dtype=float).copy()

    dimension = updated_data[function]["X"].shape[1]

    assert query.shape == (dimension,), (
        f"Function {function}: incorrect query dimension."
    )
    assert np.all(np.isfinite(query)), (
        f"Function {function}: non-finite coordinates."
    )
    assert np.all((query >= 0) & (query < 1)), (
        f"Function {function}: coordinates outside the permitted range."
    )

    query_text = "-".join(f"{value:.6f}" for value in query)

    assert re.fullmatch(
        rf"0\.\d{{6}}(?:-0\.\d{{6}}){{{dimension - 1}}}",
        query_text,
    ), f"Function {function}: invalid submission format."

    # Check the exact coordinates represented by the submission string.
    submission_query = np.asarray(
        [float(value) for value in query_text.split("-")],
        dtype=float,
    )

    assert np.array_equal(query, submission_query), (
        f"Function {function}: formatting changed the evaluated candidate."
    )

    nearest_distance = float(
        np.min(
            np.linalg.norm(
                updated_data[function]["X"] - submission_query,
                axis=1,
            )
        )
    )

    assert nearest_distance > 1e-6, (
        f"Function {function}: query duplicates an existing observation."
    )

    selected_models[function] = gp_models[function]
    selected_queries[function] = submission_query.copy()

    round_9_selected_proposals[function] = {
        **proposal,
        "query": submission_query.copy(),
        "search": selected_search,
        "query_text": query_text,
    }

    selection_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "Query": query_text,
        "Predicted mean": proposal["mean"],
        "Predicted std": proposal["std"],
        "UCB": proposal["ucb"],
        "Nearest observation distance": nearest_distance,
        "Auxiliary prediction": proposal["auxiliary_prediction"],
    })

round_9_selection = (
    pd.DataFrame(selection_rows)
    .set_index("Function")
)

display(
    round_9_selection.style.format(
        {
            "Predicted mean": "{:.6e}",
            "Predicted std": "{:.6e}",
            "UCB": "{:.6e}",
            "Nearest observation distance": "{:.6f}",
            "Auxiliary prediction": "{:.6e}",
        },
        na_rep="—",
    )
)

round_9_submission = (
    round_9_selection[["Query"]]
    .rename(columns={"Query": "Query for submission"})
)

print("\nRound 9 — Final queries for submission")

with pd.option_context("display.max_colwidth", None):
    display(round_9_submission)

# Plain-text output prevents long queries from being truncated.
for function, row in round_9_submission.iterrows():
    print(f"Function {function}: {row['Query for submission']}")

print("\nAll eight queries passed validation.")
print("Queries are prepared; this code does not submit them to the portal.")

,Selected search,Query,Predicted mean,Predicted std,UCB,Nearest observation distance,Auxiliary prediction
Function,,,,,,,
1,Global,0.257861-0.414992,-1.777581e-04,8.715443e-04,1.565331e-03,0.142548,—
2,Global,0.693314-0.998760,5.778382e-01,7.550253e-02,7.288432e-01,0.059537,—
3,Global,0.607473-0.016509-0.462216,-3.625537e-02,5.902179e-02,8.178821e-02,0.383088,—
4,Local 0.03,0.451707-0.435831-0.384426-0.407421,4.533632e-01,1.861774e-01,8.257181e-01,0.030465,—
5,Local 0.10,0.999950-0.993086-0.999495-0.943967,8.391369e+03,1.310630e+02,8.653495e+03,0.053333,7.743003e+03
6,Local 0.10,0.458107-0.374813-0.540827-0.692938-0.001140,-3.956728e-01,1.218263e-01,-1.520202e-01,0.048154,—
7,Global,0.183162-0.134864-0.733336-0.137261-0.331994-0.699922,2.294387e+00,1.189804e-01,2.532347e+00,0.247987,9.289607e-01
8,Local 0.10,0.071169-0.178617-0.100685-0.109637-0.996594-0.554316-0.240640-0.908329,9.936421e+00,2.447388e-02,9.985369e+00,0.189848,—



Round 9 — Final queries for submission


,Query for submission
Function,
1,0.257861-0.414992
2,0.693314-0.998760
3,0.607473-0.016509-0.462216
4,0.451707-0.435831-0.384426-0.407421
5,0.999950-0.993086-0.999495-0.943967
6,0.458107-0.374813-0.540827-0.692938-0.001140
7,0.183162-0.134864-0.733336-0.137261-0.331994-0.699922
8,0.071169-0.178617-0.100685-0.109637-0.996594-0.554316-0.240640-0.908329


Function 1: 0.257861-0.414992
Function 2: 0.693314-0.998760
Function 3: 0.607473-0.016509-0.462216
Function 4: 0.451707-0.435831-0.384426-0.407421
Function 5: 0.999950-0.993086-0.999495-0.943967
Function 6: 0.458107-0.374813-0.540827-0.692938-0.001140
Function 7: 0.183162-0.134864-0.733336-0.137261-0.331994-0.699922
Function 8: 0.071169-0.178617-0.100685-0.109637-0.996594-0.554316-0.240640-0.908329

All eight queries passed validation.
Queries are prepared; this code does not submit them to the portal.


### Round 9 — Final validated queries

The eight final queries were selected by maximising UCB across the global and two local search proposals for each function.

Global candidates were selected for Functions 1, 2, 3 and 7. Function 4 selected the local 0.03 proposal, while Functions 5, 6 and 8 selected the local 0.10 proposals. This distribution emerged from the candidate scores rather than a predefined allocation between global and local searches.

All queries passed checks for dimensionality, permitted coordinate range, six-decimal formatting and separation from existing observations. The coordinates displayed for submission are the same coordinates used to calculate the predictions.

Only the selected Function 8 query has a GP predicted mean above its current best observed output. For the other functions, predictive uncertainty raises the UCB above the current best, motivating evaluation despite lower predicted means. These scores do not guarantee improvement.

Auxiliary predictions were retained for Functions 5 and 7 as additional diagnostics; they did not determine the selections.

The final queries and their historical predictions are recorded in this notebook. The code prepares the queries but does not submit them to the portal. Once round 9 outputs are available, they will be compared with these predictions before the next model update.